# 3.2 Distributed lag non-linear models (confirmatory analysis of H1–H3)

Main analysis of the preregistration: *"Quasi-Poisson time series regression with distributed lag non-linear models (Gasparrini et al. 2010). Seasonality and trend are controlled with splines of time. Temperature lags extend to about 21 days, and other exposures use shorter lags."* Tests are two-sided at α = 0.05 with a **Holm correction across H1–H3**; secondary and sensitivity results are estimates with 95% confidence intervals.

The DLNM is implemented in Python (deviations log, item 21): a cross-basis is the tensor product of an exposure basis and a lag basis, as in the R package `dlnm`, fitted by a quasi-Poisson GLM (`statsmodels`, dispersion from Pearson χ²); predictions use the delta method. §2 checks the implementation on simulated arrivals with known effects, and §7 refits the primary models with R `dlnm` on the real data; §8 shows both side by side.

**Primary specification, fixed before the first fit** (deviations log, item 31):

| | H1 | H2 | H3 |
|---|---|---|---|
| Outcome | `arr_all_mts_all` | `arr_medical_mts_all` | `arr_medical_mts_all` |
| Exposure | `temp_mean` | `ili_rate` on log(1 + x) (weekly value on each day) | `pm10` |
| Exposure–response | natural cubic spline, knots at the 10th, 75th, 90th percentiles | linear | linear |
| Lag | 0–21 days, natural spline with intercept, 3 log-spaced knots | 0–14 days, natural spline, 2 log-spaced knots | 0–3 days, one coefficient per day |
| Adjustment, all | natural spline of time, 7 df per year; day of week; `holiday`; `covid` | same | same |
| Adjustment, specific | `flu_index` (log(1 + x)) | `hp_referral_era` | `hp_referral_era`; temperature cross-basis as in H1 |
| Test | F-test of all 20 cross-basis coefficients | F-test of the 4 cross-basis coefficients | F-test of the 4 lag coefficients |

**PM2.5** (secondary exposure for H3; deviations log, items 38–39) is analysed exactly like H3, with PM2.5 in place of PM10 (model `PM25`): it goes through every step below, but it is **not part of the Holm family** of H1–H3; its calibrated p-value is reported without correction.

**Residual autocorrelation** (Metodologija): if the partial autocorrelation of the deviance residuals at lag 1 exceeds ±1.96/√n in the primary model, the lag-1 deviance residual is added. This decision is made once per hypothesis, on the primary model, and **kept for all its sensitivity analyses and negative controls**, so they differ from the primary model only in the element that is varied.

**Calibration** (deviations log, items 32–33, 36): the nominal F-test is too liberal for distributed-lag cross-bases (their columns are sums over many lagged days, so small residual autocorrelations inflate it). Each hypothesis is therefore also tested against **1,000 dummy exposures** with the exposure's seasonal pattern and autocorrelation, added to the full primary model: the calibrated p-value is the share of dummy F statistics at least as large as the observed one, and the mean dummy F (κ, 1 if calibrated) widens the confidence intervals of the primary estimates by √κ. The same dummies calibrate the future-exposure negative controls and the H1 same-day and lag 1–21 tests.

All outputs are aggregated estimates. Reports go to `outputs/03_2_dlnm/`.

## Imports

In [ ]:
import hashlib
import json
import platform
import shutil
import subprocess
import warnings
from datetime import datetime, timezone
from importlib.metadata import version as pkg_version
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import patsy
import statsmodels.api as sm
from matplotlib.colors import LinearSegmentedColormap
from statsmodels.tsa.stattools import pacf
from scipy.stats import norm
norm_sf = norm.sf

## Configuration

In [ ]:
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements-lock.txt").exists())
PROCESSED = ROOT / "data" / "processed"
OUTPUTS = ROOT / "outputs" / "03_2_dlnm"
OUTPUTS.mkdir(parents=True, exist_ok=True)
PANEL_IN = PROCESSED / "daily_panel_lagged.parquet"
RUN_INFO_02 = ROOT / "outputs" / "02_lag_variables" / "run_info.json"
R_SCRIPT = ROOT / "scripts" / "validate_dlnm_r.R"
R_DIR = OUTPUTS / "r_validation"          # exchange files with R (local only, like all outputs)

ALPHA = 0.05
N_DUMMY = 1000          # dummy exposures per hypothesis (calibration; smallest calibrated p = 1/1001)
SEED = 20261109
DF_PER_YEAR = 7

# Primary specifications (fixed before the first fit)
PRIMARY = {
    "H1": dict(hyp="H1", outcome="arr_all_mts_all", exposure="temp_mean", transform=None,
               exp_basis="ns", lag_max=21, lag_basis=("ns", 3), extra=["flu_adj"], effect="temperature"),
    "H2": dict(hyp="H2", outcome="arr_medical_mts_all", exposure="ili_rate", transform="log1p",
               exp_basis="lin", lag_max=14, lag_basis=("ns", 2), extra=["hp_era"], effect="p10_p90", lead_days=7),
    "H3": dict(hyp="H3", outcome="arr_medical_mts_all", exposure="pm10", transform=None,
               exp_basis="lin", lag_max=3, lag_basis=("integer", None), extra=["hp_era", "temp_cb"], effect="per10"),
}
# Secondary exposure analysed exactly like H3 (deviations log, items 38-39); not part of the Holm family of H1-H3
SECONDARY = {
    "PM25": dict(hyp="H3, secondary exposure PM2.5", outcome="arr_medical_mts_all", exposure="pm25", transform=None,
                 exp_basis="lin", lag_max=3, lag_basis=("integer", None), extra=["hp_era", "temp_cb"], effect="per10"),
}
MODELS = {**PRIMARY, **SECONDARY}
DEFAULTS = dict(dfy=DF_PER_YEAR, covid="covid", exclude=None, resid="auto", lead=False, lead_days=None, shift=0)

COLORS = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "grey": "#8a8984", "red": "#e34948", "neutral": "#f0efec"}
DIV = LinearSegmentedColormap.from_list("div", [COLORS["blue"], COLORS["neutral"], COLORS["red"]])
plt.rcParams.update({
    "figure.figsize": (12, 3.6), "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "legend.frameon": False, "font.size": 9,
})


def fmt_p(p: float) -> str:
    return f"{p:.2g}" if p >= 1e-4 else f"{p:.1e}"

## 1. Load and check

The run stops if the panel is not the file `02_lag_variables.ipynb` last wrote.

In [ ]:
def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


run_info_02 = json.loads(RUN_INFO_02.read_text())
panel_sha = sha256(PANEL_IN)
if panel_sha != run_info_02["panel_sha256"]:
    raise RuntimeError("Panel differs from the one recorded by 02_lag_variables.ipynb; run it again.")
panel = pd.read_parquet(PANEL_IN)
YEARS = len(panel) / 365.25
# Weather covariates for sensitivity analyses: mean of the day and the day before
panel["_precip01"] = panel["precip"].rolling(2).mean()
panel["_sunshine01"] = panel["sunshine"].rolling(2).mean()
panel["_wind01"] = panel["era5_wind_mean"].rolling(2).mean()
panel["_pm_coarse"] = (panel["pm10"] - panel["pm25"]).clip(lower=0)   # coarse fraction PM10 - PM2.5 (secondary)
print(f"Panel: {len(panel)} days ({panel.index.min().date()} to {panel.index.max().date()}); checksum matches 02's run record")

## 2. Cross-basis and model functions

- `Basis`: a spline basis whose knots and boundaries are fixed on the data it was built from, so it can be evaluated at new values.
- `crossbasis`: the matrix of lagged values *x*_{t−l}, *l* = 0 … L, passed through the exposure basis and multiplied by the lag basis: column (j, k) = Σ_l B_j(x_{t−l}) C_k(l). A row is missing if any lagged value is missing.
- `fit_model`: builds the design (time spline on the analysis days, day of week, holiday, COVID adjustment, specific adjustments, cross-basis), fits the quasi-Poisson GLM on complete days, and applies the residual rule (`auto`: decide from the partial autocorrelation; `yes`/`no`: fixed).
- `predict`: log-RR at exposure *x* and lag *l* relative to a reference value, or summed over lags (cumulative), with delta-method standard errors (optionally inflated by √κ).

In [ ]:
class Basis:
    """Spline basis with knots and bounds fixed when built; evaluate with basis(x) (NaN in -> NaN out)."""

    def __init__(self, x: np.ndarray, formula: str):
        x = np.asarray(x, dtype=float)
        self.info = patsy.dmatrix(formula, {"x": x[~np.isnan(x)]}).design_info
        self.ncol = len(self.info.column_names)

    def __call__(self, x) -> np.ndarray:
        x = np.asarray(x, dtype=float)
        out = np.full((x.size, self.ncol), np.nan)
        ok = ~np.isnan(x)
        if ok.any():
            out[ok] = np.asarray(patsy.build_design_matrices([self.info], {"x": x[ok]})[0])
        return out


class Linear:
    ncol = 1

    def __call__(self, x) -> np.ndarray:
        return np.asarray(x, dtype=float).reshape(-1, 1)


def logknots(L: int, nk: int) -> list[float]:
    """Knots equally spaced on the log scale over lags 0..L (as dlnm::logknots)."""
    return [float(np.exp((1 + np.log(L)) / (nk + 1) * i - 1)) for i in range(1, nk + 1)]


def var_knots(x: np.ndarray) -> list[float]:
    return [float(v) for v in np.nanpercentile(x, [10, 75, 90])]


def dr_knots(x: np.ndarray) -> list[float]:
    """Exposure-response curves (section 14): 10th/50th/90th percentiles; of the values above the minimum if >10% sit at it (pollen)."""
    v = x[~np.isnan(x)]
    base = v[v > v.min()] if (v == v.min()).mean() > 0.1 else v
    return [float(q) for q in np.percentile(base, [10, 50, 90])]


def exposure_basis(x: np.ndarray, kind: str):
    if kind == "lin":
        return Linear()
    q = var_knots(x) if kind == "ns" else dr_knots(x)
    lo, hi = float(np.nanmin(x)), float(np.nanmax(x))
    return Basis(x, f"cr(x, knots=({q[0]!r}, {q[1]!r}, {q[2]!r}), lower_bound={lo!r}, upper_bound={hi!r}, constraints='center') - 1")


def lag_basis(L: int, kind: tuple) -> np.ndarray:
    lags = np.arange(L + 1, dtype=float)
    if kind[0] == "integer":
        return np.eye(L + 1)
    k = logknots(L, kind[1])
    return Basis(lags, f"cr(x, knots=({', '.join(repr(v) for v in k)},), lower_bound=0.0, upper_bound={float(L)!r}) - 1")(lags)


def crossbasis(x: pd.Series, eb, C: np.ndarray) -> np.ndarray:
    L = C.shape[0] - 1
    Q = np.column_stack([x.shift(l).to_numpy(dtype=float) for l in range(L + 1)])   # n x (L+1)
    B = eb(Q.ravel()).reshape(len(x), L + 1, eb.ncol)                               # n x (L+1) x vx
    return np.einsum("nlj,lk->njk", B, C).reshape(len(x), eb.ncol * C.shape[1])


def transform(s: pd.Series, t: str | None) -> pd.Series:
    if t is None:
        return s
    return np.log1p(s) if t == "log1p" else np.log(s)


def time_spline(t: np.ndarray, dfy: float) -> np.ndarray:
    df = max(3, round(dfy * len(t) / 365.25))
    return np.asarray(patsy.dmatrix(f"cr(t, df={df}, constraints='center') - 1", {"t": t}))


def lead_term(x: pd.Series, spec: dict) -> pd.DataFrame:
    """Future exposure: next week's value (weekly exposures) or the mean of the next 1-3 days."""
    if spec.get("lead_days"):
        return pd.DataFrame({"lead": x.shift(-spec["lead_days"])})
    lead = pd.concat([x.shift(-k) for k in (1, 2, 3)], axis=1).mean(axis=1, skipna=False)
    if spec["exp_basis"] == "ns":
        b = Basis(lead.to_numpy(), "cr(x, df=3, constraints='center') - 1")(lead.to_numpy())
        return pd.DataFrame(b, index=x.index, columns=[f"lead{j}" for j in range(b.shape[1])])
    return pd.DataFrame({"lead": lead})


RESID = {}   # residual decision per hypothesis, set by the primary models (section 4)


def make_spec(base: str, **over) -> dict:
    spec = {**DEFAULTS, **MODELS[base], **over}
    if "resid" not in over and base in RESID:
        spec["resid"] = "yes" if RESID[base] else "no"
    return spec


def fit_model(spec: dict, exposure_series: pd.Series | None = None) -> dict:
    """Fit one DLNM. exposure_series replaces the panel column (simulated arrivals check)."""
    y = panel[spec["outcome"]].astype(float)
    x = exposure_series if exposure_series is not None else transform(panel[spec["exposure"]].astype(float), spec["transform"])
    if spec.get("shift"):
        x = x.shift(spec["shift"])
    if spec.get("exclude") == "covid_period":
        y = y.where(panel["covid"] == 0)
    if spec.get("exclude") == "unverified_pm":
        x = x.where(panel["pm_unverified_instrument"] == 0)

    eb = exposure_basis(x.to_numpy(), spec["exp_basis"])
    C = lag_basis(spec["lag_max"], spec["lag_basis"])
    blocks = {"cb": crossbasis(x, eb, C)}
    if "temp_cb" in spec["extra"] or "temp_adj_h2" in spec["extra"]:
        t = panel[spec.get("temp_var", "temp_mean")].astype(float)
        blocks["temp_cb"] = crossbasis(t, exposure_basis(t.to_numpy(), "ns"), lag_basis(21, ("ns", 3)))
    adj = pd.DataFrame(index=panel.index)
    adj["holiday"] = panel[spec.get("holiday_var", "holiday")].astype(float)
    if spec.get("holiday_cols") is not None:          # individual holidays instead of the single indicator
        adj = adj.drop(columns="holiday").join(spec["holiday_cols"])
    if spec["covid"] in ("covid", "two_phase"):
        adj["covid"] = panel["covid"].astype(float)
    if spec["covid"] == "two_phase":
        adj["covid_restrictions"] = panel["covid_restrictions"].astype(float)
    if spec["covid"] == "stringency":
        adj["stringency"] = panel["covid_stringency_filled"].astype(float)
    if "flu_adj" in spec["extra"]:
        adj["flu_index"] = np.log1p(panel["flu_index"].astype(float))
    if "hp_era" in spec["extra"]:
        adj["hp_referral_era"] = panel["hp_referral_era"].astype(float)
    if "weather" in spec["extra"]:
        adj["precip01"], adj["sunshine01"] = panel["_precip01"], panel["_sunshine01"]
    if "wind" in spec["extra"]:
        adj["wind01"] = panel["_wind01"]
    if "holidays_ext" in spec["extra"]:
        adj["day_after_holiday"] = panel["day_after_holiday"].astype(float)
        adj["school_holiday"] = panel["school_holiday"].astype(float)
    if spec.get("lead"):
        adj = adj.join(lead_term(x, spec))
    dow = pd.get_dummies(panel["dow"].astype(int), prefix="dow", drop_first=True, dtype=float)

    def build(extra_cols: pd.DataFrame | None):
        frame = pd.concat([adj, dow] + ([extra_cols] if extra_cols is not None else []), axis=1)
        mask = y.notna().to_numpy() & frame.notna().all(axis=1).to_numpy()
        for v in blocks.values():
            mask &= ~np.isnan(v).any(axis=1)
        ts = time_spline(panel["time"].to_numpy(dtype=float)[mask], spec["dfy"])
        fr = frame.loc[mask]
        fr = fr.loc[:, fr.nunique() > 1]                       # drop indicators constant in the sample
        X = np.column_stack([np.ones(mask.sum()), ts, fr.to_numpy(), *[v[mask] for v in blocks.values()]])
        names = (["const"] + [f"time{i}" for i in range(ts.shape[1])] + list(fr.columns)
                 + [f"{k}_{i}" for k, v in blocks.items() for i in range(v.shape[1])])
        res = sm.GLM(y.to_numpy()[mask], X, family=sm.families.Poisson()).fit(scale="X2")
        return res, mask, names, fr

    res, mask, names, fr = build(None)
    r = pd.Series(np.nan, index=panel.index)
    r[mask] = res.resid_deviance
    pac1 = pacf(r.fillna(0).to_numpy(), nlags=1)[1]          # full daily index (missing days = 0)
    limit = 1.96 / np.sqrt(mask.sum())
    use_resid = spec["resid"] == "yes" or (spec["resid"] == "auto" and abs(pac1) > limit)
    if use_resid:
        res, mask, names, fr = build(pd.DataFrame({"resid_lag1": r.shift(1)}))

    idx = {k: [i for i, n in enumerate(names) if n.startswith(k + "_")] for k in blocks}
    return {"spec": spec, "res": res, "mask": mask, "names": names, "eb": eb, "C": C, "x": x, "idx": idx,
            "lead_idx": [i for i, n in enumerate(names) if n.startswith("lead")], "resid_used": use_resid,
            "pacf1_before": pac1, "pacf_limit": limit, "frame": fr}


def wald(res, R: np.ndarray) -> tuple[float, float]:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        w = res.wald_test(R, use_f=True, scalar=True)
    return float(w.statistic), float(w.pvalue)


def select(cols: list[int], k: int) -> np.ndarray:
    R = np.zeros((len(cols), k))
    R[np.arange(len(cols)), cols] = 1
    return R


def lag_split_R(m: dict, part: str) -> np.ndarray:
    """Restriction matrix for the lag-0 exposure-response (part='lag0') or the summed lags 1..L ('lag1plus')."""
    C, cols, k = m["C"], m["idx"]["cb"], len(m["res"].params)
    c = C[0] if part == "lag0" else C[1:].sum(axis=0)
    sub = np.kron(np.eye(m["eb"].ncol), c)                   # vx x (vx*vl)
    R = np.zeros((sub.shape[0], k))
    R[:, cols] = sub
    return R


def predict(m: dict, xs, ref: float, cumulative: bool = True, kappa: float = 1.0):
    """log-RR (and SE) of exposure values xs vs ref; cumulative over lags, or per lag (array lags x values)."""
    cols = m["idx"]["cb"]
    beta, V = m["res"].params[cols], m["res"].cov_params()[np.ix_(cols, cols)] * kappa
    Bx = m["eb"](np.atleast_1d(xs)) - m["eb"]([ref])
    C = m["C"]
    if cumulative:
        G = np.einsum("mj,k->mjk", Bx, C.sum(axis=0)).reshape(len(Bx), -1)
        return G @ beta, np.sqrt(np.einsum("mi,ij,mj->m", G, V, G))
    G = np.einsum("mj,lk->lmjk", Bx, C).reshape(C.shape[0], len(Bx), -1)
    return G @ beta, np.sqrt(np.einsum("lmi,ij,lmj->lm", G, V, G))


def qaic(m: dict, phi: float) -> float:
    """Quasi-AIC with a common dispersion phi (compare only models fitted on the same days)."""
    res = m["res"]
    ll = sm.families.Poisson().loglike(res.model.endog, res.mu, scale=1.0)
    return -2 * ll / phi + 2 * len(res.params)

### Implementation check on simulated arrivals (known truth)

Before any model is fitted to real arrivals, the code is checked on **simulated** daily counts with quasi-Poisson noise similar to the real data, for the three kinds of cross-basis used:

1. **Non-linear × spline lags (H1)**: the real temperature series with a J-shaped effect (cold spread over 0–10 days, heat over 0–3 days); the fitted cumulative curve should cover the true one.
2. **Linear × spline lags (H2)**: a linear effect of a simulated weekly exposure over lags 0–14, with a true lag curve that lies in the H2 lag basis (so the check tests the code, not the flexibility of the basis); lag-specific and cumulative.
3. **Linear × unconstrained lags (H3)**: a linear effect of real PM10 with known coefficients at lags 0–3.

Pass criteria: for H1, the true cumulative curve inside the 95% band on at least 90% of the temperature grid; for H2 and H3 (linear exposures, whose cross-basis coefficients have a known true value), a joint χ² test that the coefficients equal the truth gives p > 0.01. No real outcome is used here.

In [ ]:
rng_check = np.random.default_rng(1)
lags21 = np.arange(22)
w_cold = np.where(lags21 <= 10, (11 - lags21) / 66, 0)
w_heat = np.where(lags21 <= 3, (4 - lags21) / 10, 0)
def true_temp(tc, lag):
    return 0.0015 * np.minimum(tc, 0) ** 2 * w_cold[lag] + 0.003 * np.maximum(tc - 5, 0) ** 2 * w_heat[lag]


def simulate_counts(lin: pd.Series) -> pd.Series:
    season = 0.1 * np.sin(2 * np.pi * panel["time"] / 365.25)
    mu = np.exp(np.log(200) + season + lin).fillna(200)
    phi = 2.0
    y = pd.Series(rng_check.negative_binomial(mu / (phi - 1), 1 / phi), index=panel.index, dtype=float)
    return y.where(lin.notna())


from scipy.stats import chi2


def coef_test(m: dict, truth: np.ndarray) -> float:
    """Joint chi-square test that the cross-basis coefficients equal the true ones (linear exposures)."""
    cols = m["idx"]["cb"]
    d = m["res"].params[cols] - truth
    return float(chi2.sf(d @ np.linalg.solve(m["res"].cov_params()[np.ix_(cols, cols)], d), len(cols)))


checks = {}
# 1. H1-type
temp = panel["temp_mean"].astype(float)
lin = sum(true_temp(temp.shift(l) - 15, l) for l in lags21)
panel["_y_sim"] = simulate_counts(lin)
m_c = fit_model(make_spec("H1", outcome="_y_sim", extra=[], covid="none", resid="no"))
grid = np.linspace(np.nanpercentile(temp, 1), np.nanpercentile(temp, 99), 100)
truth = np.array([sum(true_temp(g - 15, l) for l in lags21) for g in grid]) - sum(true_temp(0.0, l) for l in lags21)
est, se = predict(m_c, grid, ref=15.0)
checks["non-linear x spline lags (H1)"] = (np.mean(np.abs(est - truth) <= 1.96 * se), np.max(np.abs(est - truth)), np.nan)
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.fill_between(grid, np.exp(est - 1.96 * se), np.exp(est + 1.96 * se), color=COLORS["blue"], alpha=0.2, lw=0, label="estimate, 95% CI")
ax.plot(grid, np.exp(est), color=COLORS["blue"], lw=2)
ax.plot(grid, np.exp(truth), color=COLORS["orange"], lw=2, ls="--", label="true")
ax.axhline(1, color="black", lw=0.6)
ax.set_xlabel("temperature (°C)"); ax.set_ylabel("cumulative RR vs 15 °C")
ax.set_title("Implementation check, H1-type cross-basis", loc="left"); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUTPUTS / "implementation_check.png"); plt.show()

# 2. H2-type: weekly exposure, linear, lags 0-14 with a smooth bump (true lag weights)
wk = pd.Series(rng_check.gamma(2, 1, size=400), index=pd.period_range(panel.index[0], periods=400, freq="W-SUN"))
xw = pd.Series(wk.reindex(panel.index.to_period("W-SUN")).to_numpy(), index=panel.index)
C2 = lag_basis(14, ("ns", 2))                                   # true lag curve inside the H2 lag basis
g2 = np.array([0.004, 0.012, 0.003, -0.002])
lw = C2 @ g2
panel["_y_sim"] = simulate_counts(sum(lw[l] * xw.shift(l) for l in range(15)))
m_c = fit_model(make_spec("H2", outcome="_y_sim", extra=[], covid="none", resid="no"), exposure_series=xw)
e, s = predict(m_c, [1.0], ref=0.0, cumulative=False)
checks["linear x spline lags (H2)"] = (np.mean(np.abs(e[:, 0] - lw) <= 1.96 * s[:, 0]), np.max(np.abs(e[:, 0] - lw)), coef_test(m_c, g2))

# 3. H3-type: real PM10, linear, unconstrained lags 0-3
pm = panel["pm10"].astype(float)
b3 = np.array([0.004, 0.002, 0.001, 0.0])
panel["_y_sim"] = simulate_counts(sum(b3[l] * pm.shift(l) for l in range(4)))
m_c = fit_model(make_spec("H3", outcome="_y_sim", extra=[], covid="none", resid="no"))
e, s = predict(m_c, [1.0], ref=0.0, cumulative=False)
checks["linear x unconstrained lags (H3)"] = (np.mean(np.abs(e[:, 0] - b3) <= 1.96 * s[:, 0]), np.max(np.abs(e[:, 0] - b3)), coef_test(m_c, b3))
del panel["_y_sim"]

chk = pd.DataFrame(checks, index=["share of truth inside 95% CI", "max |error| (log-RR)", "joint test p (coefficients = truth)"]).T
display(chk.round(4))
if chk.iloc[0, 0] < 0.9 or (chk["joint test p (coefficients = truth)"].dropna() < 0.01).any():
    raise RuntimeError("Implementation check failed: a cross-basis type does not recover the simulated effect")

## 3. Effect summaries

- **H1**: the cumulative exposure–response curve (lags 0–21) over the 1st–99th percentile range. Prespecified: RR at the 99th (heat) and 1st (cold) percentile against the minimum-arrivals temperature (MAT). Because the MAT can fall at the edge of the range (then the heat RR is 1 by construction), RRs against the **median** temperature are also given, cumulative and on the same day (lag 0). The F-test is split into the **same-day** exposure–response (lag 0, 4 df) and the **summed lags 1–21** (4 df).
- **H2**: cumulative RR for an increase from the 10th to the 90th percentile of ILI (log(1 + x) scale), lags 0–14.
- **H3**: cumulative RR per 10 µg/m³ PM10 over lags 0–3 (per 10th→90th percentile when PM10 is on the log scale).

`kappa` widens the confidence intervals by √κ (κ from the dummy exposures, §6; 1 = nominal).

In [ ]:
def summarize(m: dict, kappa: float = 1.0) -> dict:
    spec, x = m["spec"], m["x"]
    xs = x[m["mask"]]
    k = len(m["res"].params)
    F, p = wald(m["res"], select(m["idx"]["cb"], k))
    out = {"n_days": int(m["mask"].sum()), "dispersion": m["res"].scale, "resid_term": m["resid_used"], "F": F, "p": p}
    ci = lambda e, s: np.exp([e, e - 1.96 * s, e + 1.96 * s])
    if spec["effect"] == "temperature":
        p1, p99 = np.nanpercentile(xs, [1, 99])
        grid = np.linspace(p1, p99, 200)
        med = float(np.nanmedian(xs))
        est, _ = predict(m, grid, ref=med)
        mat = float(grid[np.argmin(est)])
        out.update(MAT=mat, median_temp=med, heat_value=p99, cold_value=p1)
        for name, val in (("heat", p99), ("cold", p1)):
            e, s = predict(m, [val], ref=mat, kappa=kappa)
            out[f"RR_{name}"], out[f"lo_{name}"], out[f"hi_{name}"] = ci(e[0], s[0])
            e, s = predict(m, [val], ref=med, kappa=kappa)
            out[f"RR_{name}_vs_median"], out[f"lo_{name}_vs_median"], out[f"hi_{name}_vs_median"] = ci(e[0], s[0])
            e, s = predict(m, [val], ref=med, cumulative=False, kappa=kappa)
            out[f"RR_{name}_lag0"], out[f"lo_{name}_lag0"], out[f"hi_{name}_lag0"] = ci(e[0, 0], s[0, 0])
        out["F_lag0"], out["p_lag0"] = wald(m["res"], lag_split_R(m, "lag0"))
        out["F_lag1plus"], out["p_lag1plus"] = wald(m["res"], lag_split_R(m, "lag1plus"))
    else:
        if spec["effect"] == "per10" and spec["transform"] is None:
            lo_v, hi_v, label = 0.0, 10.0, "per 10 µg/m³"
        else:
            lo_v, hi_v = np.nanpercentile(xs, [10, 90])
            label = "10th to 90th percentile"
        e, s = predict(m, [hi_v], ref=lo_v, kappa=kappa)
        out["RR"], out["lo"], out["hi"] = ci(e[0], s[0])
        out["increment"] = label
    if m["lead_idx"]:
        out["F_lead"], out["p_lead"] = wald(m["res"], select(m["lead_idx"], k))
    return out

## 4. Primary models (nominal tests)

The residual decision of each primary model is stored and reused for all its variants. The nominal p-values here are **not** the final confirmatory results; §6 calibrates them; §8 gives the final results.

In [ ]:
primary = {}
for h in MODELS:
    primary[h] = fit_model(make_spec(h))
    RESID[h] = primary[h]["resid_used"]
prim = pd.DataFrame({h: {**summarize(m), "pacf1 before residual term": m["pacf1_before"], "pacf limit": m["pacf_limit"]}
                     for h, m in primary.items()}).T
for h, m in primary.items():
    print(f"{h}: n = {int(m['mask'].sum())} days, dispersion {m['res'].scale:.2f}, lag-1 partial autocorrelation "
          f"{m['pacf1_before']:.3f} (limit {m['pacf_limit']:.3f}) -> residual term {'yes' if m['resid_used'] else 'no'}; "
          f"nominal F = {prim.loc[h, 'F']:.2f}, p = {fmt_p(prim.loc[h, 'p'])}")

### Model checks

Partial autocorrelation of the deviance residuals of the primary models, and residuals over time.

In [ ]:
fig, axes = plt.subplots(2, len(primary), figsize=(16, 6))
for j, (h, m) in enumerate(primary.items()):
    r = pd.Series(np.nan, index=panel.index)
    r[m["mask"]] = m["res"].resid_deviance
    pc = pacf(r.fillna(0).to_numpy(), nlags=21)
    lim = 1.96 / np.sqrt(m["mask"].sum())
    ax = axes[0, j]
    ax.bar(range(1, 22), pc[1:], color=COLORS["blue"], width=0.6)
    ax.axhline(lim, ls="--", color=COLORS["grey"], lw=1)
    ax.axhline(-lim, ls="--", color=COLORS["grey"], lw=1)
    ax.set_title(f"{h}: PACF of deviance residuals", loc="left")
    ax.set_xlabel("lag (days)")
    ax = axes[1, j]
    ax.plot(r.index, r, lw=0.4, color=COLORS["grey"])
    ax.plot(r.index, r.rolling(28, center=True, min_periods=20).mean(), color=COLORS["blue"], lw=1.5)
    ax.set_title(f"{h}: deviance residuals (28-day mean)", loc="left")
fig.tight_layout()
fig.savefig(OUTPUTS / "model_checks.png")
plt.show()

## 5. Negative controls and future exposure (nominal)

1. **Outcome**: H2 and H3 with injuries by unit (`arr_injury_mts_all`) and by diagnosis (`arr_dx_injury_icd`).
2. **Future exposure**: the primary model plus the mean exposure of the next 1–3 days (H2: next week's value); `p_lead` tests the future term. These p-values are calibrated in §6.

In [ ]:
nc_models = {
    "H2 → injuries (KPP)": fit_model(make_spec("H2", outcome="arr_injury_mts_all")),
    "H2 → injuries (ICD S/T)": fit_model(make_spec("H2", outcome="arr_dx_injury_icd")),
    "H3 → injuries (KPP)": fit_model(make_spec("H3", outcome="arr_injury_mts_all")),
    "H3 → injuries (ICD S/T)": fit_model(make_spec("H3", outcome="arr_dx_injury_icd")),
    "PM25 → injuries (KPP)": fit_model(make_spec("PM25", outcome="arr_injury_mts_all")),
    "PM25 → injuries (ICD S/T)": fit_model(make_spec("PM25", outcome="arr_dx_injury_icd")),
}
lead_models = {h: fit_model(make_spec(h, lead=True)) for h in MODELS}
nc = pd.DataFrame({k: summarize(m) for k, m in nc_models.items()}).T
leads = pd.DataFrame({f"{h} + future exposure": summarize(m) for h, m in lead_models.items()}).T

## 6. Dummy exposures: calibrated tests and confidence intervals

For each hypothesis, `N_DUMMY` dummy exposures keep the exposure's seasonal pattern (its fit on the time spline) and autocorrelation (an AR(*p*) model of the remaining variation, *p* by AIC; influenza simulated per ISO week), with the same missing days, but are unrelated to arrivals. Each dummy's cross-basis **and** its future-exposure term are added to the final primary model, and tested with the same F-tests as the real exposure:

- **Dummy share p < 0.05**: the preregistered negative control; about 5% if the nominal test is calibrated.
- **Calibrated p** = (1 + number of dummy F ≥ observed F) / (N_DUMMY + 1), for the main test, the H1 same-day and lag 1–21 tests, and the future-exposure test. Holm correction across H1–H3 on the main tests.
- **κ** = mean dummy F of the main test (1 if calibrated); the confidence intervals of the primary estimates are widened by √κ.

In [ ]:
def ar_fit(v: np.ndarray, max_p: int) -> tuple[np.ndarray, float]:
    best = None
    for p in range(1, max_p + 1):
        L = np.column_stack([np.r_[np.full(j, np.nan), v[:-j]] for j in range(1, p + 1)])
        ok = ~np.isnan(v) & ~np.isnan(L).any(axis=1)
        A = np.column_stack([np.ones(ok.sum()), L[ok]])
        beta, *_ = np.linalg.lstsq(A, v[ok], rcond=None)
        rss = np.sum((v[ok] - A @ beta) ** 2)
        aic = ok.sum() * np.log(rss / ok.sum()) + 2 * (p + 1)
        if best is None or aic < best[0]:
            best = (aic, beta[1:], np.sqrt(rss / (ok.sum() - p - 1)))
    return best[1], best[2]


def simulate_ar(phi: np.ndarray, sigma: float, n: int, rng) -> np.ndarray:
    p, burn = len(phi), 300
    e = rng.normal(0, sigma, n + burn)
    v = np.zeros(n + burn)
    for t in range(p, n + burn):
        v[t] = e[t] + phi @ v[t - p:t][::-1]
    return v[burn:]


def dummy_generator(spec: dict, weekly: bool = False):
    """Return a function rng -> dummy series with the exposure's seasonal fit and AR structure."""
    x = transform(panel[spec["exposure"]].astype(float), spec["transform"])
    if weekly:   # weekly exposure (ILI): simulate per ISO week
        w = x.groupby(panel.index.to_period("W-SUN")).first()
        t = np.arange(len(w), dtype=float)
        ok = w.notna().to_numpy()
        S = np.asarray(patsy.dmatrix(f"cr(t, df={max(3, round(DF_PER_YEAR * len(w) / 52.18))}, constraints='center')", {"t": t}))
        beta, *_ = np.linalg.lstsq(S[ok], w.to_numpy()[ok], rcond=None)
        phi, sigma = ar_fit(np.where(ok, w.to_numpy() - S @ beta, np.nan), 4)
        week_of_day = panel.index.to_period("W-SUN")
        def gen(rng):
            d = pd.Series(np.where(ok, S @ beta + simulate_ar(phi, sigma, len(w), rng), np.nan), index=w.index)
            return pd.Series(d.reindex(week_of_day).to_numpy(), index=panel.index)
        return gen
    t = panel["time"].to_numpy(dtype=float)
    ok = x.notna().to_numpy()
    S = np.asarray(patsy.dmatrix(f"cr(t, df={round(DF_PER_YEAR * YEARS)}, constraints='center')", {"t": t}))
    beta, *_ = np.linalg.lstsq(S[ok], x.to_numpy()[ok], rcond=None)
    phi, sigma = ar_fit(np.where(ok, x.to_numpy() - S @ beta, np.nan), 14)
    return lambda rng: pd.Series(np.where(ok, S @ beta + simulate_ar(phi, sigma, len(x), rng), np.nan), index=panel.index)


def dummy_stats(m: dict, d: pd.Series) -> dict:
    """Add the dummy's cross-basis and future term to the final primary model; F-tests as for the real exposure."""
    spec = m["spec"]
    eb = exposure_basis(d.to_numpy(), spec["exp_basis"])
    C = lag_basis(spec["lag_max"], spec["lag_basis"])
    cbd = crossbasis(d, eb, C)[m["mask"]]
    ld = lead_term(d, spec).to_numpy()[m["mask"]]
    ok = ~np.isnan(cbd).any(axis=1) & ~np.isnan(ld).any(axis=1)
    X0 = m["res"].model.exog
    X = np.column_stack([X0, cbd, ld])[ok]
    r = sm.GLM(m["res"].model.endog[ok], X, family=sm.families.Poisson()).fit(scale="X2")
    k0, kc, k = X0.shape[1], cbd.shape[1], X.shape[1]
    out = {}
    out["F"], out["p"] = wald(r, select(list(range(k0, k0 + kc)), k))
    out["F_lead"], _ = wald(r, select(list(range(k0 + kc, k)), k))
    if spec["effect"] == "temperature":
        for part in ("lag0", "lag1plus"):
            c = C[0] if part == "lag0" else C[1:].sum(axis=0)
            sub = np.kron(np.eye(eb.ncol), c)
            R = np.zeros((sub.shape[0], k)); R[:, k0:k0 + kc] = sub
            out[f"F_{part}"], _ = wald(r, R)
    return out


rng = np.random.default_rng(SEED)
dummy = {}
for h in MODELS:
    gen = dummy_generator(MODELS[h], weekly=(h == "H2"))
    dummy[h] = pd.DataFrame([dummy_stats(primary[h], gen(rng)) for _ in range(N_DUMMY)])
pd.concat(dummy, names=["hypothesis", "draw"]).to_csv(OUTPUTS / "dummy_exposure.csv")

calp = lambda Fd, Fobs: (1 + (Fd >= Fobs).sum()) / (len(Fd) + 1)
KAPPA = {h: float(dummy[h]["F"].mean()) for h in MODELS}
binom = np.random.default_rng(1).binomial(N_DUMMY, 0.05, 100000) / N_DUMMY
print(f"Dummy share p < 0.05 should lie within {np.percentile(binom, 2.5):.3f}–{np.percentile(binom, 97.5):.3f} if the nominal test is calibrated.")
display(pd.DataFrame({"dummy share p < 0.05": {h: (dummy[h]["p"] < 0.05).mean() for h in MODELS},
                      "kappa (mean dummy F)": KAPPA,
                      "CI widening factor sqrt(kappa)": {h: np.sqrt(k) for h, k in KAPPA.items()}}).round(3))

fig, axes = plt.subplots(1, len(MODELS), figsize=(15, 3.4))
for ax, h in zip(axes, MODELS):
    pv = np.sort(dummy[h]["p"].to_numpy())
    ex = (np.arange(1, len(pv) + 1) - 0.5) / len(pv)
    ax.plot([0, 1], [0, 1], color=COLORS["grey"], lw=1)
    ax.scatter(ex, pv, s=4, color=COLORS["blue"])
    ax.set_title(f"{h}: dummy exposures, QQ of nominal p", loc="left")
    ax.set_xlabel("expected p"); ax.set_ylabel("observed p")
fig.tight_layout(); fig.savefig(OUTPUTS / "dummy_exposure_qq.png"); plt.show()

## 7. Validation against R `dlnm`

The design of each primary model (outcome, the exposure series, and all non-exposure columns exactly as used here) is exported to `outputs/03_2_dlnm/r_validation/`. `scripts/validate_dlnm_r.R` builds the cross-basis **independently** with `dlnm::crossbasis` (same knots, lag ranges and `logknots`), fits the same quasi-Poisson model with `glm`, and writes `dlnm::crosspred` predictions and the Wald F statistic. The predictions should agree to numerical precision (natural cubic splines with the same knots span the same space, so the fitted curves are identical even though the bases differ).

Runs only if `Rscript` is installed (with the packages `dlnm` and `splines`): install R from <https://cran.r-project.org>, then in R `install.packages("dlnm")`.

In [ ]:
R_DIR.mkdir(parents=True, exist_ok=True)
for h, m in primary.items():
    spec = m["spec"]
    rows = np.flatnonzero(m["mask"])
    names, X = m["names"], m["res"].model.exog
    keep = [i for i, n in enumerate(names) if not (n.startswith("cb_") or n.startswith("temp_cb_") or n == "const")]
    conf = pd.DataFrame(np.full((len(panel), len(keep)), np.nan), index=panel.index, columns=[f"z{i}" for i in range(len(keep))])
    conf.iloc[rows] = X[:, keep]
    out = pd.DataFrame({"y": np.nan, "in_sample": 0, "x": m["x"].to_numpy(),
                        "temp": panel["temp_mean"].to_numpy()}, index=panel.index)
    out.iloc[rows, 0] = m["res"].model.endog
    out.iloc[rows, 1] = 1
    pd.concat([out, conf], axis=1).to_csv(R_DIR / f"{h}_data.csv", index_label="date")
    xs = m["x"][m["mask"]]
    if spec["effect"] == "temperature":
        at, cen = np.linspace(np.nanpercentile(xs, 1), np.nanpercentile(xs, 99), 25), float(np.nanmedian(xs))
    else:
        lo_v, hi_v = (0.0, 10.0) if spec["transform"] is None else tuple(np.nanpercentile(xs, [10, 90]))
        at, cen = np.array([hi_v]), lo_v
    e, s = predict(m, at, ref=cen)
    F, pF = wald(m["res"], select(m["idx"]["cb"], len(m["res"].params)))
    meta = {"exp_basis": spec["exp_basis"], "lag_max": spec["lag_max"], "lag_basis": spec["lag_basis"][0],
            "lag_nk": spec["lag_basis"][1], "var_knots": var_knots(m["x"].to_numpy()) if spec["exp_basis"] == "ns" else None,
            "var_range": [float(np.nanmin(m["x"])), float(np.nanmax(m["x"]))],
            "temp_cb": "temp_cb" in spec["extra"], "temp_knots": var_knots(panel["temp_mean"].to_numpy()),
            "temp_range": [float(panel["temp_mean"].min()), float(panel["temp_mean"].max())],
            "at": at.tolist(), "cen": cen}
    (R_DIR / f"{h}_meta.json").write_text(json.dumps(meta))
    pd.DataFrame({"at": at, "logRR_python": e, "se_python": s, "F_python": F, "p_python": pF}).to_csv(R_DIR / f"{h}_python.csv", index=False)

rscript = shutil.which("Rscript")
if rscript is None:
    print("Rscript not found: validation skipped. Install R and the dlnm package, then run this notebook again "
          f"(or run: Rscript {R_SCRIPT.relative_to(ROOT)} {R_DIR.relative_to(ROOT)}).")
    r_validation = None
else:
    res_r = subprocess.run([rscript, str(R_SCRIPT), str(R_DIR)], capture_output=True, text=True, timeout=600)
    print(res_r.stdout[-2000:], res_r.stderr[-2000:])
    rows = []
    for h in MODELS:
        py = pd.read_csv(R_DIR / f"{h}_python.csv")
        rr = pd.read_csv(R_DIR / f"{h}_r.csv")
        rows.append({"hypothesis": h, "max |logRR diff|": np.max(np.abs(py["logRR_python"] - rr["logRR_r"])),
                     "max |se diff| / se": np.max(np.abs(py["se_python"] - rr["se_r"]) / py["se_python"]),
                     "F python": py["F_python"].iloc[0], "F R": rr["F_r"].iloc[0],
                     "p python": py["p_python"].iloc[0], "p R": rr["p_r"].iloc[0]})
    r_validation = pd.DataFrame(rows).set_index("hypothesis")
    r_validation.to_csv(OUTPUTS / "r_validation.csv")
    display(r_validation)

## 8. Confirmatory results

**The final result of each hypothesis is the calibrated, Holm-corrected p-value** (Holm across H1–H3; PM2.5, secondary, calibrated p without correction); estimates have √κ-widened 95% confidence intervals. Nominal values are shown for comparison.

**Python vs R `dlnm`**: the same models refitted with R `dlnm` (§7) give the same estimates, confidence intervals, F statistics and p-values; the table and figure below show both side by side (nominal intervals in both, so they are directly comparable).

In [ ]:
def holm(p: pd.Series) -> pd.Series:
    order, out, run = p.sort_values().index, {}, 0.0
    for i, h in enumerate(order):
        run = max(run, min(1.0, (len(order) - i) * p[h]))
        out[h] = run
    return pd.Series(out)[p.index]


final = pd.DataFrame({h: summarize(primary[h], kappa=KAPPA[h]) for h in MODELS}).T
final["p calibrated"] = [calp(dummy[h]["F"], final.loc[h, "F"]) for h in MODELS]
final["p nominal, Holm"] = holm(final.loc[list(PRIMARY), "p"].astype(float))            # Holm family: H1-H3 only
final["p calibrated, Holm"] = holm(final.loc[list(PRIMARY), "p calibrated"].astype(float))
final["reject H0 (calibrated, Holm, α = 0.05)"] = (final["p calibrated, Holm"] < ALPHA).where(final.index.isin(list(PRIMARY)))
final["kappa"] = pd.Series(KAPPA)
final.loc["H1", "p_lag0 calibrated"] = calp(dummy["H1"]["F_lag0"], final.loc["H1", "F_lag0"])
final.loc["H1", "p_lag1plus calibrated"] = calp(dummy["H1"]["F_lag1plus"], final.loc["H1", "F_lag1plus"])
final.to_csv(OUTPUTS / "primary_results.csv")

r = final.loc["H1"]
print(f"H1 temperature -> all arrivals: calibrated Holm p = {fmt_p(r['p calibrated, Holm'])} (nominal {fmt_p(r['p'])}).")
print(f"   same day (lag 0): calibrated p = {fmt_p(r['p_lag0 calibrated'])}; summed lags 1-21: calibrated p = {fmt_p(r['p_lag1plus calibrated'])}")
print(f"   same day vs median {r['median_temp']:.1f} °C: heat ({r['heat_value']:.1f} °C) RR {r['RR_heat_lag0']:.3f} ({r['lo_heat_lag0']:.3f}–{r['hi_heat_lag0']:.3f}), "
      f"cold ({r['cold_value']:.1f} °C) RR {r['RR_cold_lag0']:.3f} ({r['lo_cold_lag0']:.3f}–{r['hi_cold_lag0']:.3f})")
print(f"   cumulative 0-21 vs median: heat RR {r['RR_heat_vs_median']:.3f} ({r['lo_heat_vs_median']:.3f}–{r['hi_heat_vs_median']:.3f}), "
      f"cold RR {r['RR_cold_vs_median']:.3f} ({r['lo_cold_vs_median']:.3f}–{r['hi_cold_vs_median']:.3f}); "
      f"vs MAT {r['MAT']:.1f} °C (prespecified): heat {r['RR_heat']:.3f}, cold {r['RR_cold']:.3f} ({r['lo_cold']:.3f}–{r['hi_cold']:.3f})")
for h in ("H2", "H3"):
    r = final.loc[h]
    print(f"{h}: RR {r['RR']:.3f} ({r['lo']:.3f}–{r['hi']:.3f}), {r['increment']}; calibrated Holm p = {fmt_p(r['p calibrated, Holm'])} "
          f"(nominal {fmt_p(r['p'])})")
r = final.loc["PM25"]
print(f"PM2.5 (H3, secondary exposure): RR {r['RR']:.3f} ({r['lo']:.3f}–{r['hi']:.3f}), {r['increment']}; calibrated p = "
      f"{fmt_p(r['p calibrated'])} (nominal {fmt_p(r['p'])}; not in the Holm family)")
display(final[["n_days", "resid_term", "kappa", "F", "p", "p calibrated", "p nominal, Holm", "p calibrated, Holm",
               "reject H0 (calibrated, Holm, α = 0.05)"]])

# Python vs R dlnm, same model (nominal 95% CIs in both, so they are directly comparable)
if r_validation is None:
    print("R validation not run (Rscript not installed): see section 7.")
else:
    ci = lambda e, s: f"{np.exp(e):.4f} ({np.exp(e - 1.96 * s):.4f}–{np.exp(e + 1.96 * s):.4f})"
    rows = []
    for h in MODELS:
        py, rr = pd.read_csv(R_DIR / f"{h}_python.csv"), pd.read_csv(R_DIR / f"{h}_r.csv")
        picks = ([("cold (1st pct) vs median, cumulative 0–21", 0), ("heat (99th pct) vs median, cumulative 0–21", len(py) - 1)]
                 if h == "H1" else [(f"cumulative, {final.loc[h, 'increment']}", 0)])
        for label, i in picks:
            rows.append({"hypothesis": h, "estimate": label,
                         "Python RR (95% CI)": ci(py["logRR_python"][i], py["se_python"][i]),
                         "R dlnm RR (95% CI)": ci(rr["logRR_r"][i], rr["se_r"][i]),
                         "|log-RR difference|": f"{abs(py['logRR_python'][i] - rr['logRR_r'][i]):.1e}",
                         "F Python": round(py["F_python"][0], 4), "F R": round(rr["F_r"][0], 4),
                         "p Python (nominal)": fmt_p(py["p_python"][0]), "p R (nominal)": fmt_p(rr["p_r"][0])})
    print("\nPython implementation vs R dlnm (same data and model; nominal 95% CIs):")
    display(pd.DataFrame(rows).set_index(["hypothesis", "estimate"]))

    fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), gridspec_kw={"width_ratios": [2, 1, 1, 1]})
    py, rr = pd.read_csv(R_DIR / "H1_python.csv"), pd.read_csv(R_DIR / "H1_r.csv")
    ax = axes[0]
    ax.fill_between(py["at"], np.exp(py["logRR_python"] - 1.96 * py["se_python"]), np.exp(py["logRR_python"] + 1.96 * py["se_python"]),
                    color=COLORS["blue"], alpha=0.2, lw=0)
    ax.plot(py["at"], np.exp(py["logRR_python"]), color=COLORS["blue"], lw=2, label="Python")
    ax.scatter(rr["at"], np.exp(rr["logRR_r"]), color=COLORS["orange"], s=18, zorder=3, label="R dlnm")
    ax.axhline(1, color="black", lw=0.6)
    ax.set_xlabel("temperature (°C)"); ax.set_ylabel("RR vs median, cumulative 0–21")
    ax.set_title("H1: Python vs R dlnm", loc="left"); ax.legend(fontsize=7)
    for ax, h in zip(axes[1:], ("H2", "H3", "PM25")):
        py, rr = pd.read_csv(R_DIR / f"{h}_python.csv"), pd.read_csv(R_DIR / f"{h}_r.csv")
        for k, (d, col, lab) in enumerate(((py, COLORS["blue"], "Python"), (rr, COLORS["orange"], "R dlnm"))):
            e, s = d.iloc[0, 1], d.iloc[0, 2]
            ax.errorbar([k], [np.exp(e)], yerr=[[np.exp(e) - np.exp(e - 1.96 * s)], [np.exp(e + 1.96 * s) - np.exp(e)]],
                        fmt="o", color=col, capsize=0, lw=1.5)
        ax.axhline(1, color="black", lw=0.6)
        ax.set_xticks([0, 1], ["Python", "R dlnm"]); ax.set_xlim(-0.6, 1.6)
        ax.set_ylabel(f"RR, {final.loc[h, 'increment']}"); ax.set_title(f"{h}: Python vs R dlnm", loc="left")
    fig.tight_layout(); fig.savefig(OUTPUTS / "python_vs_r.png"); plt.show()

# Negative controls with calibrated future-exposure tests
leads["p_lead calibrated"] = [calp(dummy[h]["F_lead"], leads.loc[f"{h} + future exposure", "F_lead"]) for h in MODELS]
neg = pd.concat([nc, leads])
neg.to_csv(OUTPUTS / "negative_controls.csv")
print("\nNegative controls (outcome controls: nominal CIs; the H3 and PM2.5 tests are calibrated, H2's is liberal by kappa above):")
cols = [c for c in ["RR", "lo", "hi", "increment", "p", "p_lead", "p_lead calibrated", "n_days"] if c in neg]
display(neg[cols])

## 9. Figures of the primary associations

Bands are √κ-widened 95% confidence intervals. **H1**: cumulative exposure–response (lags 0–21) against the median with the temperature distribution; lag–response at the 1st and 99th percentiles; RR over temperature × lag. **H2, H3, PM2.5**: lag-specific RR.

In [ ]:
m, r1, kap = primary["H1"], final.loc["H1"], KAPPA["H1"]
xs = m["x"][m["mask"]]
ref = r1["median_temp"]
grid = np.linspace(np.nanpercentile(xs, 0.5), np.nanpercentile(xs, 99.5), 200)
est, se = predict(m, grid, ref=ref, kappa=kap)
fig = plt.figure(figsize=(13, 7.5))
ax = fig.add_subplot(2, 2, 1)
ax.fill_between(grid, np.exp(est - 1.96 * se), np.exp(est + 1.96 * se), color=COLORS["blue"], alpha=0.2, lw=0)
ax.plot(grid, np.exp(est), color=COLORS["blue"], lw=2)
ax.axhline(1, color="black", lw=0.6)
for v, lab in ((r1["cold_value"], "1st pct"), (ref, "median"), (r1["heat_value"], "99th pct")):
    ax.axvline(v, color=COLORS["grey"], ls=":", lw=1)
    ax.text(v, ax.get_ylim()[1], lab, fontsize=7, ha="center", va="bottom")
ax2 = ax.twinx()
ax2.hist(xs, bins=60, color=COLORS["grey"], alpha=0.25)
ax2.set_yticks([]); ax2.grid(False); ax2.spines["right"].set_visible(False)
ax.set_xlabel("daily mean temperature (°C)")
ax.set_ylabel("RR vs median, cumulative over lags 0–21")
ax.set_title("H1: temperature and all ED arrivals", loc="left")

ax = fig.add_subplot(2, 2, 2)
for val, lab, col in ((r1["heat_value"], f"99th pct ({r1['heat_value']:.1f} °C)", COLORS["red"]),
                      (r1["cold_value"], f"1st pct ({r1['cold_value']:.1f} °C)", COLORS["blue"])):
    e, s = predict(m, [val], ref=ref, cumulative=False, kappa=kap)
    lg = np.arange(len(e))
    ax.fill_between(lg, np.exp(e[:, 0] - 1.96 * s[:, 0]), np.exp(e[:, 0] + 1.96 * s[:, 0]), color=col, alpha=0.15, lw=0)
    ax.plot(lg, np.exp(e[:, 0]), color=col, lw=2, label=lab)
ax.axhline(1, color="black", lw=0.6)
ax.set_xlabel("lag (days)"); ax.set_ylabel("RR vs median")
ax.set_title("H1: lag–response", loc="left"); ax.legend(fontsize=7)

ax = fig.add_subplot(2, 1, 2)
tg = np.linspace(np.nanpercentile(xs, 1), np.nanpercentile(xs, 99), 80)
e, _ = predict(m, tg, ref=ref, cumulative=False)
v = np.nanmax(np.abs(e))
im = ax.pcolormesh(tg, np.arange(e.shape[0]), np.exp(e), cmap=DIV, vmin=np.exp(-v), vmax=np.exp(v), shading="nearest")
ax.set_xlabel("temperature (°C)"); ax.set_ylabel("lag (days)")
ax.set_title("H1: RR by temperature and lag (vs median)", loc="left"); ax.grid(False)
fig.colorbar(im, ax=ax, label="RR")
fig.tight_layout(); fig.savefig(OUTPUTS / "H1_temperature.png"); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, h in zip(axes, ("H2", "H3", "PM25")):
    m, r = primary[h], final.loc[h]
    xs = m["x"][m["mask"]]
    lo_v, hi_v = (0.0, 10.0) if m["spec"]["transform"] is None else tuple(np.nanpercentile(xs, [10, 90]))
    e, s = predict(m, [hi_v], ref=lo_v, cumulative=False, kappa=KAPPA[h])
    lg = np.arange(len(e))
    ax.fill_between(lg, np.exp(e[:, 0] - 1.96 * s[:, 0]), np.exp(e[:, 0] + 1.96 * s[:, 0]), color=COLORS["blue"], alpha=0.2, lw=0)
    ax.plot(lg, np.exp(e[:, 0]), color=COLORS["blue"], lw=2, marker="o" if len(lg) < 6 else None)
    ax.axhline(1, color="black", lw=0.6)
    ax.set_xlabel("lag (days)"); ax.set_ylabel(f"RR, {r['increment']}")
    ptxt = f"calibrated Holm p = {fmt_p(r['p calibrated, Holm'])}" if h in PRIMARY else f"calibrated p = {fmt_p(r['p calibrated'])} (secondary)"
    ax.set_title(f"{h}: {m['spec']['exposure']} → {m['spec']['outcome']}\ncumulative RR {r['RR']:.3f} "
                 f"({r['lo']:.3f}–{r['hi']:.3f}), {ptxt}", loc="left", fontsize=8)
fig.tight_layout(); fig.savefig(OUTPUTS / "H2_H3_PM25_lag_response.png"); plt.show()

### Lag-specific estimates (saved for the comparison with the cross-correlation, `03_3_comparison.ipynb`)

RR at each lag (√κ-widened 95% CI): H1 at the 99th and 1st percentile vs the median; H2 for the 10th→90th percentile increase; H3 per 10 µg/m³.

In [ ]:
rows = []
for h, m in primary.items():
    xs = m["x"][m["mask"]]
    if h == "H1":
        r1 = final.loc["H1"]
        contrasts = [("heat (99th pct vs median)", r1["heat_value"], r1["median_temp"]), ("cold (1st pct vs median)", r1["cold_value"], r1["median_temp"])]
    else:
        lo_v, hi_v = (0.0, 10.0) if m["spec"]["transform"] is None else tuple(np.nanpercentile(xs, [10, 90]))
        contrasts = [(final.loc[h, "increment"], hi_v, lo_v)]
    for label, val, ref in contrasts:
        e, s = predict(m, [val], ref=ref, cumulative=False, kappa=KAPPA[h])
        for lag in range(e.shape[0]):
            rows.append({"hypothesis": h, "contrast": label, "lag_days": lag, "RR": np.exp(e[lag, 0]),
                         "lo": np.exp(e[lag, 0] - 1.96 * s[lag, 0]), "hi": np.exp(e[lag, 0] + 1.96 * s[lag, 0])})
lag_response = pd.DataFrame(rows)
lag_response.to_csv(OUTPUTS / "lag_response.csv", index=False)
print(f"Saved {(OUTPUTS / 'lag_response.csv').relative_to(ROOT)} ({len(lag_response)} rows)")

## 10. Secondary: H1 by stream

The H1 model fitted to each arrival stream (stream-level models add `hp_referral_era`). Secondary analysis: estimates with nominal 95% CIs (the dummy calibration was done for the all-arrivals model only).

In [ ]:
streams = {"injuries (KPP)": "arr_injury_mts_all", "medical (EzB + INP)": "arr_medical_mts_all", "fast track (HP)": "arr_fasttrack_mts_all"}
by_stream = pd.DataFrame({k: summarize(fit_model(make_spec("H1", outcome=o, extra=["flu_adj", "hp_era"]))) for k, o in streams.items()}).T
by_stream.to_csv(OUTPUTS / "H1_by_stream.csv")
show = ["RR_heat_lag0", "lo_heat_lag0", "hi_heat_lag0", "RR_cold_lag0", "lo_cold_lag0", "hi_cold_lag0",
        "RR_heat_vs_median", "lo_heat_vs_median", "hi_heat_vs_median", "RR_cold_vs_median", "lo_cold_vs_median", "hi_cold_vs_median", "p"]
display(by_stream[show].astype(float).round(3))

## 11. PM2.5: additional analyses

PM2.5 runs through every step above like H1–H3 (model `PM25`: residual rule, dummy calibration, R validation, figures, negative controls, future exposure) and through all H3 sensitivity analyses (§15) and the exposure–response curves (§14). Two further models: PM2.5 against **all arrivals**, and the **coarse fraction** (PM10 − PM2.5), which separates coarse from fine particles (PM10 and PM2.5 correlate at r = 0.90). Nominal 95% CIs.

In [ ]:
pm25_extra = pd.DataFrame({
    "PM2.5 → all arrivals": summarize(fit_model(make_spec("PM25", outcome="arr_all_mts_all"))),
    "Coarse PM (PM10 − PM2.5) → medical": summarize(fit_model(make_spec("PM25", exposure="_pm_coarse"))),
}).T
pm25_extra.to_csv(OUTPUTS / "pm25_extra.csv")
display(pm25_extra[["RR", "lo", "hi", "increment", "p", "n_days"]])

## 12. Exploratory: hospital occupancy, Martinovo, moon and pollen

**Not preregistered** (deviations log, item 40). Same confounder control as the DLNM (time spline 7 df/year, day of week, holidays, COVID period, temperature cross-basis as in H1, `hp_referral_era`), linear exposure with one coefficient per lag:

| Exposure | Lags | Contrast | Outcomes |
|---|---|---|---|
| Hospital occupancy (mean patients in UKC Maribor, 2021 – Oct 2024) | previous days 1–7 (today's arrivals cannot cause it) | +50 patients | all, medical, injuries, fast track, admissions |
| Martinovo (11 Nov; 5 days) | day and day after; holiday indicator without Martinovo | Martinovo vs ordinary day | all, medical, injuries, fast track |
| Full moon, full moon ± 1 day, new moon (negative-control exposures) | same day | event vs other days | all, medical, injuries, fast track |
| Pollen, six series, log(1 + grains/m³) (2021–2023) | 0–3 days | 90th percentile of pollen days vs no pollen | medical, respiratory diagnoses (ICD-10 J), all, injuries (control) |

**Calibration**: for each exposure, `N_DUMMY_EXPL` dummy exposures (continuous exposures: seasonal pattern and autocorrelation as in §6; events: the same number of event days placed on random days) are added to the model of its main outcome (first in the list). κ (mean dummy F) widens the CIs of all outcomes of that exposure by √κ and gives κ-adjusted p-values (F/κ); the main outcome also gets the empirical calibrated p. With about 40 tests, roughly two will have p < 0.05 by chance.

In [ ]:
from scipy.stats import f as f_dist

N_DUMMY_EXPL = 200
OUT4 = {"all": "arr_all_mts_all", "medical": "arr_medical_mts_all", "injuries": "arr_injury_mts_all", "fast track": "arr_fasttrack_mts_all"}
POLLEN = ["pollen_total", "pollen_poaceae", "pollen_betula", "pollen_ambrosia", "pollen_urticaceae", "pollen_cupressaceae"]
POLLEN_OUT = {"medical": "arr_medical_mts_all", "respiratory dx (J)": "arr_dx_resp", "all": "arr_all_mts_all", "injuries (control)": "arr_injury_mts_all"}


def xspec(exposure, outcome, lag_max, transform=None, shift=0, **over):
    return {**DEFAULTS, "hyp": "exploratory", "exposure": exposure, "outcome": outcome, "transform": transform, "exp_basis": "lin",
            "lag_max": lag_max, "lag_basis": ("integer", None), "extra": ["temp_cb", "hp_era"], "effect": "custom",
            "shift": shift, "resid": "auto", **over}


def event_dummy(x: pd.Series, rng) -> pd.Series:
    ok = x.notna().to_numpy()
    k = int(np.nansum(x.to_numpy() > 0))
    d = np.zeros(len(x))
    d[rng.choice(np.flatnonzero(ok), size=k, replace=False)] = 1.0
    return pd.Series(np.where(ok, d, np.nan), index=x.index)


EXPLORE = {   # name: (exposure, outcomes, lag_max, transform, shift, kind, contrast (hi, lo) or None = pollen rule, extra spec)
    "Hospital occupancy": ("hosp_occupancy", {**OUT4, "admissions": "adm_all"}, 6, None, 1, "continuous", (50.0, 0.0), {}),
    "Martinovo": ("martinovo", OUT4, 1, None, 0, "event", (1.0, 0.0), {"holiday_var": "holiday_workfree"}),
    "Full moon": ("full_moon", OUT4, 0, None, 0, "event", (1.0, 0.0), {}),
    "Full moon ± 1 day": ("full_moon_window", OUT4, 0, None, 0, "event", (1.0, 0.0), {}),
    "New moon": ("new_moon", OUT4, 0, None, 0, "event", (1.0, 0.0), {}),
    **{p.replace("pollen_", "Pollen: "): (p, POLLEN_OUT, 3, "log1p", 0, "continuous", None, {}) for p in POLLEN},
}

rng_x = np.random.default_rng(SEED + 40)
rows, lag_rows, kappas = [], [], {}
for name, (expo, outs, L, tr, sh, kind, contrast, over) in EXPLORE.items():
    if contrast is None:   # pollen: 90th percentile of pollen days vs no pollen (log scale)
        contrast = (float(np.nanpercentile(np.log1p(panel.loc[panel[expo] > 0, expo]), 90)), 0.0)
    models = {o: fit_model(xspec(expo, col, L, tr, sh, **over)) for o, col in outs.items()}
    main = next(iter(models.values()))
    if kind == "continuous":
        gen = dummy_generator(xspec(expo, outs[next(iter(outs))], L, tr, sh, **over))
    else:
        x0 = panel[expo].astype(float)
        gen = lambda rng, x0=x0: event_dummy(x0, rng)
    dF = np.array([dummy_stats(main, gen(rng_x))["F"] for _ in range(N_DUMMY_EXPL)])
    kap = float(dF.mean()); kappas[name] = kap
    for k, (o, m) in enumerate(models.items()):
        F, p = wald(m["res"], select(m["idx"]["cb"], len(m["res"].params)))
        q = len(m["idx"]["cb"])
        e, s_ = predict(m, [contrast[0]], ref=contrast[1], kappa=kap)
        rows.append({"exposure": name, "outcome": o, "n_days": int(m["mask"].sum()), "kappa": kap,
                     "RR": np.exp(e[0]), "lo": np.exp(e[0] - 1.96 * s_[0]), "hi": np.exp(e[0] + 1.96 * s_[0]),
                     "p nominal": p, "p kappa-adjusted": float(f_dist.sf(F / kap, q, m["res"].df_resid)),
                     "p calibrated (main outcome)": calp(dF, F) if k == 0 else np.nan,
                     "contrast": "+50 patients" if expo == "hosp_occupancy" else (f"{np.expm1(contrast[0]):.0f} grains/m³ vs 0" if tr == "log1p" else "event vs none")})
        el, sl = predict(m, [contrast[0]], ref=contrast[1], cumulative=False, kappa=kap)
        for l in range(el.shape[0]):
            lag_rows.append({"exposure": name, "outcome": o, "lag": l + sh, "RR": np.exp(el[l, 0]),
                             "lo": np.exp(el[l, 0] - 1.96 * sl[l, 0]), "hi": np.exp(el[l, 0] + 1.96 * sl[l, 0])})
explore = pd.DataFrame(rows)
explore_lags = pd.DataFrame(lag_rows)
explore.to_csv(OUTPUTS / "exploratory_exposures.csv", index=False)
explore_lags.to_csv(OUTPUTS / "exploratory_exposures_lags.csv", index=False)
print("kappa per exposure:", {k: round(v, 2) for k, v in kappas.items()})
with pd.option_context("display.max_rows", 80, "display.width", 250):
    display(explore.set_index(["exposure", "outcome"])[["contrast", "RR", "lo", "hi", "p nominal", "p kappa-adjusted", "p calibrated (main outcome)", "n_days"]].round(4))

In [ ]:
groups = [("Hospital occupancy", ["Hospital occupancy"], "RR per +50 patients, days 1–7 before"),
          ("Martinovo", ["Martinovo"], "RR, Martinovo and the day after"),
          ("Moon (negative control)", ["Full moon", "Full moon ± 1 day", "New moon"], "RR, same day"),
          ("Pollen", [n for n in EXPLORE if n.startswith("Pollen")], "RR, high pollen day vs none, lags 0–3")]
OUT_COL = {"all": COLORS["grey"], "medical": COLORS["blue"], "injuries": COLORS["orange"], "injuries (control)": COLORS["orange"],
           "fast track": COLORS["aqua"], "admissions": "#6250d6", "respiratory dx (J)": "#6250d6"}
fig, axes = plt.subplots(2, 2, figsize=(15, 12), gridspec_kw={"height_ratios": [1, 2.2], "wspace": 0.55, "hspace": 0.25})
for ax, (title, names, xlabel) in zip(axes.flat, groups):
    sub_ = explore[explore["exposure"].isin(names)].reset_index(drop=True)
    y = np.arange(len(sub_))[::-1]
    for yi, (_, r) in zip(y, sub_.iterrows()):
        c = OUT_COL.get(r["outcome"], COLORS["grey"])
        ax.errorbar([r["RR"]], [yi], xerr=[[r["RR"] - r["lo"]], [r["hi"] - r["RR"]]], fmt="o", color=c, ms=4, lw=1.3)
        if r["p kappa-adjusted"] < 0.05:
            ax.text(r["hi"], yi, "  *", va="center", fontsize=9)
    ax.axvline(1, color="black", lw=0.6)
    labels = [f"{r['exposure'].replace('Pollen: ', '')} – {r['outcome']}" if len(names) > 1 else r["outcome"] for _, r in sub_.iterrows()]
    ax.set_yticks(y, labels, fontsize=7)
    ax.set_title(title, loc="left"); ax.set_xlabel(xlabel, fontsize=8)
fig.suptitle("Exploratory exposures (√κ-widened 95% CI; * κ-adjusted p < 0.05; orange = injuries)", x=0.01, ha="left")
fig.savefig(OUTPUTS / "exploratory_exposures.png", bbox_inches="tight"); plt.show()

# Lag pattern of the exposures with a kappa-adjusted p < 0.05 for a non-control outcome
hits = explore[(explore["p kappa-adjusted"] < 0.05) & ~explore["outcome"].str.contains("injur")]
if len(hits):
    fig, axes = plt.subplots(1, len(hits), figsize=(4.2 * len(hits), 3.3), squeeze=False)
    for ax, (_, r) in zip(axes[0], hits.iterrows()):
        g = explore_lags[(explore_lags["exposure"] == r["exposure"]) & (explore_lags["outcome"] == r["outcome"])]
        ax.errorbar(g["lag"], g["RR"], yerr=[g["RR"] - g["lo"], g["hi"] - g["RR"]], fmt="o-", color=COLORS["blue"], ms=4, lw=1.2)
        ax.axhline(1, color="black", lw=0.6)
        ax.set_xlabel("lag (days)"); ax.set_ylabel("RR")
        ax.set_title(f"{r['exposure']} → {r['outcome']}", loc="left", fontsize=8)
    fig.tight_layout(); fig.savefig(OUTPUTS / "exploratory_lags.png"); plt.show()

## 13. Exploratory: each holiday separately

**Not preregistered** (deviations log, item 41). The single `holiday` indicator of the H1 model is replaced by one indicator per holiday (all its days; Easter Sunday and Monday together, New Year and Labour Day both days) and one for the **first ordinary day after** it. Consecutive holidays share that day (Christmas and Independence Day: 27 Dec; Reformation Day and All Saints' Day: 2 Nov). Martinovo (not work-free) and the Day of Solidarity (14 Aug 2023, one-off) are included. Each coefficient compares the day with ordinary days of the **same weekday**, adjusted for temperature (H1 cross-basis), influenza, the time spline and the COVID period. Holiday names come from the official list (`data/raw/calendar/prazniki_uradni.csv`, checked by the data manifest in notebook 01). Each holiday occurs 5–6 times (Day of Solidarity once), so CIs are wide; nominal 95% CIs (single-day indicators are close to calibrated: κ ≈ 1 for Martinovo and the moon in §12).

In [ ]:
NAMES = {"novo leto": "New Year (1–2 Jan)", "Prešernov dan, slovenski kulturni praznik": "Prešeren Day (8 Feb)",
         "velika noč": "Easter (Sun–Mon)", "velikonočni ponedeljek": "Easter (Sun–Mon)", "dan boja proti okupatorju": "Day of Uprising (27 Apr)",
         "praznik dela": "Labour Day (1–2 May)", "binkoštna nedelja": "Whit Sunday", "dan državnosti": "Statehood Day (25 Jun)",
         "Marijino vnebovzetje": "Assumption (15 Aug)", "dan reformacije": "Reformation Day (31 Oct)", "dan spomina na mrtve": "All Saints' Day (1 Nov)",
         "božič": "Christmas (25 Dec)", "dan samostojnosti in enotnosti": "Independence Day (26 Dec)"}
off = pd.read_csv(ROOT / "data" / "raw" / "calendar" / "prazniki_uradni.csv", sep=";")
off["date"] = pd.to_datetime(off["DATUM"], format="%d.%m.%Y")
off = off[(off["DELA_PROST_DAN"] == "da") & off["date"].between(panel.index.min(), panel.index.max())]
hol_name = pd.Series(off["IME_PRAZNIKA"].str.strip().map(NAMES).to_numpy(), index=off["date"])   # raw names may carry trailing spaces
for d in panel.index[(panel["holiday_workfree"] == 1) & ~panel.index.isin(hol_name.index)]:
    hol_name[d] = "Day of Solidarity (14 Aug 2023)" if (d.month, d.day) == (8, 14) else f"other ({d.date()})"
for d in panel.index[panel["martinovo"] == 1]:
    hol_name[d] = "Martinovo (11 Nov)"
hol_name = hol_name.sort_index()
if hol_name.isna().any() or set(hol_name.index) != set(panel.index[panel["holiday"] == 1]):
    raise RuntimeError("Holiday names do not match the panel's holiday indicator")

cols = pd.DataFrame(0.0, index=panel.index, columns=[f"on: {n}" for n in hol_name.unique()])
for d, n in hol_name.items():
    cols.loc[d, f"on: {n}"] = 1.0
is_hol = panel.index.isin(hol_name.index)
for i, d in enumerate(panel.index[:-1]):
    if is_hol[i] and not is_hol[i + 1]:                    # last day of a run of holidays -> next day is "after"
        col = f"after: {hol_name[d]}"
        if col not in cols:
            cols[col] = 0.0
        cols.loc[panel.index[i + 1], col] = 1.0
print(f"{len(hol_name)} holiday days, {hol_name.nunique()} holidays; columns: {cols.shape[1]}")

hol_rows = []
for o, col in OUT4.items():
    m = fit_model(make_spec("H1", outcome=col, extra=["flu_adj", "hp_era"], holiday_cols=cols))
    V = m["res"].cov_params()
    for c in cols.columns:
        if c not in m["names"]:
            continue
        i = m["names"].index(c)
        b, se = m["res"].params[i], np.sqrt(V[i, i])
        kind, name = c.split(": ", 1)
        hol_rows.append({"holiday": name, "day": kind, "outcome": o, "n_days": int(cols.loc[m["mask"], c].sum()),
                         "RR": np.exp(b), "lo": np.exp(b - 1.96 * se), "hi": np.exp(b + 1.96 * se), "p": 2 * norm_sf(abs(b / se))})
holidays = pd.DataFrame(hol_rows)
holidays.to_csv(OUTPUTS / "holidays_individual.csv", index=False)
with pd.option_context("display.max_rows", 200, "display.width", 250):
    display(holidays.pivot_table(index=["holiday", "day"], columns="outcome", values="RR", sort=False).round(3))

In [ ]:
order = [n for n in dict.fromkeys(hol_name.sort_index().map(lambda n: n))]
order = sorted(order, key=lambda n: hol_name[hol_name == n].index.map(lambda d: (d.month, d.day)).min())
fig, axes = plt.subplots(1, 4, figsize=(18, 6.5), sharey=True, gridspec_kw={"wspace": 0.08})
for ax, o in zip(axes, OUT4):
    sub_ = holidays[holidays["outcome"] == o]
    for k, (kind, col, dy) in enumerate((("on", COLORS["blue"], 0.15), ("after", COLORS["orange"], -0.15))):
        s2 = sub_[sub_["day"] == kind].set_index("holiday").reindex(order)
        y = np.arange(len(order))[::-1] + dy
        ax.errorbar(s2["RR"], y, xerr=[s2["RR"] - s2["lo"], s2["hi"] - s2["RR"]], fmt="o", color=col, ms=3.5, lw=1.1,
                    label="holiday" if kind == "on" else "first day after")
    ax.axvline(1, color="black", lw=0.6)
    ax.set_xscale("log"); ax.set_title(o, loc="left"); ax.set_xlabel("RR vs ordinary day, same weekday")
    from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter
    ax.xaxis.set_major_locator(FixedLocator([0.6, 0.7, 0.8, 0.9, 1.0, 1.2, 1.4, 1.6]))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
    ax.xaxis.set_minor_formatter(NullFormatter())
axes[0].set_yticks(np.arange(len(order))[::-1], order, fontsize=7.5)
axes[0].legend(fontsize=7, loc="lower left")
fig.suptitle("Each holiday separately (nominal 95% CI)", x=0.01, ha="left")
fig.savefig(OUTPUTS / "holidays_individual.png", bbox_inches="tight"); plt.show()

## 14. Exposure–response (dose–response) curves

**Not preregistered** (deviations log, item 42). The primary models of H2, H3 and PM2.5 assume a **linear** exposure–response on the analysis scale. Here each continuous exposure is refitted with a **natural cubic spline** for the exposure (knots at the 10th, 50th and 90th percentiles; for pollen, of the days with pollen), keeping the lag structure, confounders and residual decision of the linear model. Each curve is the cumulative RR over all lags against a reference value, with the linear model's line for comparison and the exposure distribution underneath.

- **Nonlinearity test**: quasi-likelihood F-test of the spline against the linear model (the linear model is nested in the spline; same days). Nominal p.
- **Confidence bands**: nominal 95% CI widened by √κ of the corresponding linear model (§6, §12). κ was estimated for the linear cross-basis, so the widening is approximate.
- **H1** is already nonlinear (primary spline); its curve is repeated here for each arrival stream.

| Exposure → outcome | Reference | Lags |
|---|---|---|
| Temperature → all arrivals and each stream | median | 0–21 |
| ILI → medical (H2) | 10th percentile | 0–14 |
| PM10 → medical (H3), PM2.5 → medical | 10th percentile | 0–3 |
| PM10, PM2.5 → injuries (negative control) | 10th percentile | 0–3 |
| Hospital occupancy → all arrivals, admissions | median | previous days 1–7 |
| Pollen (total) → medical; nettle pollen → respiratory diagnoses | no pollen | 0–3 |

In [ ]:
def dr_pair(spec_lin: dict, kind: str = "ns_dr") -> dict:
    """Linear and spline exposure-response with the same days, confounders and residual decision."""
    m_lin = fit_model(spec_lin)
    m_ns = fit_model({**spec_lin, "exp_basis": kind, "resid": "yes" if m_lin["resid_used"] else "no"})
    if not (m_lin["mask"] == m_ns["mask"]).all():
        raise RuntimeError("linear and spline models were fitted on different days")
    rl, rn = m_lin["res"], m_ns["res"]
    ddf = len(rn.params) - len(rl.params)
    F_nl = max(rl.deviance - rn.deviance, 0.0) / ddf / rn.scale
    F_all, p_all = wald(rn, select(m_ns["idx"]["cb"], len(rn.params)))
    return {"lin": m_lin, "ns": m_ns, "F_nonlin": F_nl, "df_nonlin": ddf,
            "p_nonlin": float(f_dist.sf(F_nl, ddf, rn.df_resid)), "F_spline": F_all, "p_spline": p_all}


inj = "arr_injury_mts_all"
occ = EXPLORE["Hospital occupancy"]
DR = {   # name: (linear spec, reference rule, kappa, x label)
    "H1: temperature → all arrivals": (make_spec("H1", exp_basis="lin"), "median", KAPPA["H1"], "daily mean temperature (°C)"),
    "H2: ILI → medical": (make_spec("H2"), "p10", KAPPA["H2"], "ILI rate per 100,000 (log scale)"),
    "H3: PM10 → medical": (make_spec("H3"), "p10", KAPPA["H3"], "PM10 (µg/m³)"),
    "PM2.5 → medical": (make_spec("PM25"), "p10", KAPPA["PM25"], "PM2.5 (µg/m³)"),
    "PM10 → injuries (control)": (make_spec("H3", outcome=inj), "p10", KAPPA["H3"], "PM10 (µg/m³)"),
    "PM2.5 → injuries (control)": (make_spec("PM25", outcome=inj), "p10", KAPPA["PM25"], "PM2.5 (µg/m³)"),
    "Occupancy → all arrivals": (xspec(occ[0], "arr_all_mts_all", occ[2], occ[3], occ[4]), "median", kappas["Hospital occupancy"], "patients in hospital, previous days"),
    "Occupancy → admissions": (xspec(occ[0], "adm_all", occ[2], occ[3], occ[4]), "median", kappas["Hospital occupancy"], "patients in hospital, previous days"),
    "Pollen (total) → medical": (xspec("pollen_total", "arr_medical_mts_all", 3, "log1p"), "zero", kappas["Pollen: total"], "pollen grains/m³ (log scale)"),
    "Nettle pollen → respiratory dx": (xspec("pollen_urticaceae", "arr_dx_resp", 3, "log1p"), "zero", kappas["Pollen: urticaceae"], "nettle pollen grains/m³ (log scale)"),
}

dr_fits, dr_rows, dr_tests = {}, [], []
for name, (spec, ref_rule, kap, xlab) in DR.items():
    f = dr_pair(spec, kind="ns" if name.startswith("H1") else "ns_dr")
    if name.startswith("H1"):                     # H1: the primary spline itself
        f["ns"] = primary["H1"]
    m = f["ns"]
    xs = m["x"][m["mask"]].to_numpy(dtype=float)
    ref = {"median": np.nanmedian(xs), "p10": np.nanpercentile(xs, 10), "zero": 0.0}[ref_rule]
    grid = np.linspace(np.nanpercentile(xs, 1), np.nanpercentile(xs, 99), 150)
    if ref_rule == "zero":
        grid = np.linspace(0.0, np.nanpercentile(xs, 99.5), 150)
    f.update(ref=float(ref), grid=grid, kappa=kap, xlab=xlab)
    dr_fits[name] = f
    for model in ("ns", "lin"):
        e, s = predict(f[model], grid, ref=ref, kappa=kap)
        back = np.expm1 if f["ns"]["spec"]["transform"] == "log1p" else (lambda v: v)
        dr_rows += [{"curve": name, "model": "spline" if model == "ns" else "linear", "exposure_value": float(back(g)),
                     "reference": float(back(ref)), "RR": np.exp(a), "lo": np.exp(a - 1.96 * b), "hi": np.exp(a + 1.96 * b)}
                    for g, a, b in zip(grid, e, s)]
    dr_tests.append({"curve": name, "n_days": int(m["mask"].sum()), "kappa": kap, "F spline (overall)": f["F_spline"],
                     "p spline (overall, nominal)": f["p_spline"], "F nonlinearity": f["F_nonlin"], "df nonlinearity": f["df_nonlin"],
                     "p nonlinearity (nominal)": f["p_nonlin"]})
dose_response = pd.DataFrame(dr_rows)
dose_response_tests = pd.DataFrame(dr_tests).set_index("curve")
dose_response.to_csv(OUTPUTS / "dose_response.csv", index=False)
dose_response_tests.to_csv(OUTPUTS / "dose_response_tests.csv")
display(dose_response_tests.round(4))

In [ ]:
def dr_panel(ax, f, title):
    m = f["ns"]
    tr = m["spec"]["transform"] == "log1p"
    back = np.expm1 if tr else (lambda v: v)
    xs = m["x"][m["mask"]].to_numpy(dtype=float)
    gx = back(f["grid"])
    e, s = predict(m, f["grid"], ref=f["ref"], kappa=f["kappa"])
    ax.fill_between(gx, np.exp(e - 1.96 * s), np.exp(e + 1.96 * s), color=COLORS["blue"], alpha=0.2, lw=0)
    ax.plot(gx, np.exp(e), color=COLORS["blue"], lw=2, label="spline")
    el, _ = predict(f["lin"], f["grid"], ref=f["ref"])
    ax.plot(gx, np.exp(el), color=COLORS["orange"], lw=1.3, ls="--", label="linear")
    ax.axhline(1, color="black", lw=0.6)
    ax.axvline(back(f["ref"]), color=COLORS["grey"], ls=":", lw=1)
    ax2 = ax.twinx()
    bins = np.linspace(f["grid"].min(), f["grid"].max(), 40)
    ax2.hist(back(xs[(xs >= bins[0]) & (xs <= bins[-1])]), bins=back(bins), color=COLORS["grey"], alpha=0.25)
    ax2.set_yticks([]); ax2.grid(False); ax2.spines["right"].set_visible(False)
    ax2.set_ylim(0, ax2.get_ylim()[1] * 3)                   # histogram in the bottom third
    if tr:
        ax.set_xscale("symlog", linthresh=1.0); ax2.set_xscale("symlog", linthresh=1.0)
    ax.set_xlabel(f["xlab"], fontsize=8); ax.set_ylabel("cumulative RR vs reference", fontsize=8)
    ax.set_title(f"{title}\nnonlinearity p = {fmt_p(f['p_nonlin'])}, spline overall p = {fmt_p(f['p_spline'])} (nominal)",
                 loc="left", fontsize=8)


names = list(DR)
ncol = 3
fig, axes = plt.subplots(int(np.ceil(len(names) / ncol)), ncol, figsize=(15, 3.9 * np.ceil(len(names) / ncol)))
for ax, n in zip(axes.flat, names):
    dr_panel(ax, dr_fits[n], n)
for ax in list(axes.flat)[len(names):]:
    ax.set_visible(False)
axes.flat[0].legend(fontsize=7, loc="upper center")
fig.suptitle("Exposure–response curves, cumulative over all lags (√κ-widened 95% CI; dotted line = reference; grey = exposure distribution)",
             x=0.01, ha="left")
fig.tight_layout(); fig.savefig(OUTPUTS / "dose_response.png", bbox_inches="tight"); plt.show()

# H1 by stream: the primary temperature spline fitted to each stream
streams_all = {"all arrivals": "arr_all_mts_all", **streams}
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8), sharey=True)
st_rows = []
for ax, (lab, o) in zip(axes, streams_all.items()):
    m = primary["H1"] if o == "arr_all_mts_all" else fit_model(make_spec("H1", outcome=o, extra=["flu_adj", "hp_era"]))
    xs = m["x"][m["mask"]].to_numpy(dtype=float)
    med = float(np.nanmedian(xs))
    grid = np.linspace(np.nanpercentile(xs, 1), np.nanpercentile(xs, 99), 150)
    kap = KAPPA["H1"] if o == "arr_all_mts_all" else 1.0
    e, s = predict(m, grid, ref=med, kappa=kap)
    ax.fill_between(grid, np.exp(e - 1.96 * s), np.exp(e + 1.96 * s), color=COLORS["blue"], alpha=0.2, lw=0)
    ax.plot(grid, np.exp(e), color=COLORS["blue"], lw=2)
    e0, s0 = predict(m, grid, ref=med, cumulative=False, kappa=kap)
    ax.plot(grid, np.exp(e0[0]), color=COLORS["red"], lw=1.3, ls="--", label="same day (lag 0)")
    ax.axhline(1, color="black", lw=0.6)
    ax.set_title(lab, loc="left"); ax.set_xlabel("daily mean temperature (°C)")
    st_rows += [{"stream": lab, "temperature": g, "RR cumulative": np.exp(a), "lo": np.exp(a - 1.96 * b), "hi": np.exp(a + 1.96 * b),
                 "RR lag 0": np.exp(c)} for g, a, b, c in zip(grid, e, s, e0[0])]
axes[0].set_ylabel("RR vs median temperature"); axes[0].legend(fontsize=7)
fig.suptitle("H1: temperature–response by stream (blue: cumulative lags 0–21 with 95% CI, √κ-widened for all arrivals only; red: same day)",
             x=0.01, ha="left")
fig.tight_layout(); fig.savefig(OUTPUTS / "H1_dose_response_by_stream.png", bbox_inches="tight"); plt.show()
pd.DataFrame(st_rows).to_csv(OUTPUTS / "H1_dose_response_by_stream.csv", index=False)

## 15. Sensitivity analyses

Each row changes one element of the primary model; the residual decision of the primary model is kept. Confidence intervals are nominal (for H1 and H2 they are too narrow by about √κ, §6). New in this version (deviations log, item 36): adjustment for other weather (precipitation, sunshine; for H3 also wind), day-after-holiday and school holidays, and for H2 the previous week's ILI only (no within-week look-ahead).

The time-spline choice is checked by QAIC on the **same days** with a **common dispersion** (from the 10 df/year model).

In [ ]:
sens_specs = {
    "H1 primary": make_spec("H1"),
    "H1 time 4 df/year": make_spec("H1", dfy=4), "H1 time 10 df/year": make_spec("H1", dfy=10),
    "H1 lag 0–14": make_spec("H1", lag_max=14), "H1 lag 0–28": make_spec("H1", lag_max=28),
    "H1 no residual term": make_spec("H1", resid="no"),
    "H1 excluding COVID period": make_spec("H1", exclude="covid_period", covid="none"),
    "H1 two-phase COVID": make_spec("H1", covid="two_phase"),
    "H1 stringency index": make_spec("H1", covid="stringency"),
    "H1 no influenza adjustment": make_spec("H1", extra=[]),
    "H1 + precipitation, sunshine": make_spec("H1", extra=["flu_adj", "weather"]),
    "H1 + day after holiday, school holidays": make_spec("H1", extra=["flu_adj", "holidays_ext"]),
    "H1 Tmax": make_spec("H1", exposure="temp_max"), "H1 Tmin": make_spec("H1", exposure="temp_min"),
    "H1 ERA5-Land temperature": make_spec("H1", exposure="era5_temp_mean"),
    "H1 apparent temperature": make_spec("H1", exposure="era5_apparent_temp_mean"),
    "H1 episodes, 4 h window": make_spec("H1", outcome="arr_all_mts_all_w4h"),
    "H1 episodes, 24 h window": make_spec("H1", outcome="arr_all_mts_all_w24h"),
    "H2 primary": make_spec("H2"),
    "H2 time 4 df/year": make_spec("H2", dfy=4), "H2 time 10 df/year": make_spec("H2", dfy=10),
    "H2 lag 0–7": make_spec("H2", lag_max=7, lag_basis=("ns", 1)), "H2 lag 0–21": make_spec("H2", lag_max=21, lag_basis=("ns", 2)),
    "H2 previous week's ILI only": make_spec("H2", shift=7),
    "H2 no residual term": make_spec("H2", resid="no"),
    "H2 raw scale": make_spec("H2", transform=None),
    "H2 influenza index (2021–2026)": make_spec("H2", exposure="flu_index"),
    "H2 + temperature": make_spec("H2", extra=["hp_era", "temp_adj_h2"]),
    "H2 + day after holiday, school holidays": make_spec("H2", extra=["hp_era", "holidays_ext"]),
    "H2 Enota za bolezni only": make_spec("H2", outcome="arr_disease_mts_all"),
    "H2 INP only": make_spec("H2", outcome="arr_inp_mts_all"),
    "H2 two-phase COVID": make_spec("H2", covid="two_phase"),
    "H3 primary": make_spec("H3"),
    "H3 time 4 df/year": make_spec("H3", dfy=4), "H3 time 10 df/year": make_spec("H3", dfy=10),
    "H3 lag 0–7": make_spec("H3", lag_max=7, lag_basis=("ns", 1)),
    "H3 no residual term": make_spec("H3", resid="no"),
    "H3 no temperature adjustment": make_spec("H3", extra=["hp_era"]),
    "H3 + precipitation, sunshine, wind": make_spec("H3", extra=["hp_era", "temp_cb", "weather", "wind"]),
    "H3 + day after holiday, school holidays": make_spec("H3", extra=["hp_era", "temp_cb", "holidays_ext"]),
    "H3 excluding COVID period": make_spec("H3", exclude="covid_period", covid="none"),
    "H3 verified PM10 only": make_spec("H3", exclude="unverified_pm"),
    "H3 PM10 Titova (traffic)": make_spec("H3", exposure="pm10_titova"),
    "H3 PM10 Vrbanski (ARSO hourly)": make_spec("H3", exposure="pm10_vrbanski"),
    "H3 log scale": make_spec("H3", transform="log"),
    "H3 Enota za bolezni only": make_spec("H3", outcome="arr_disease_mts_all"),
    "H3 INP only": make_spec("H3", outcome="arr_inp_mts_all"),
    "PM25 primary": make_spec("PM25"),
    "PM25 time 4 df/year": make_spec("PM25", dfy=4), "PM25 time 10 df/year": make_spec("PM25", dfy=10),
    "PM25 lag 0–7": make_spec("PM25", lag_max=7, lag_basis=("ns", 1)),
    "PM25 no residual term": make_spec("PM25", resid="no"),
    "PM25 no temperature adjustment": make_spec("PM25", extra=["hp_era"]),
    "PM25 + precipitation, sunshine, wind": make_spec("PM25", extra=["hp_era", "temp_cb", "weather", "wind"]),
    "PM25 + day after holiday, school holidays": make_spec("PM25", extra=["hp_era", "temp_cb", "holidays_ext"]),
    "PM25 excluding COVID period": make_spec("PM25", exclude="covid_period", covid="none"),
    "PM25 verified PM2.5 only": make_spec("PM25", exclude="unverified_pm"),
    "PM25 PM2.5 Titova (traffic)": make_spec("PM25", exposure="pm25_titova"),
    "PM25 PM2.5 Vrbanski (ARSO hourly)": make_spec("PM25", exposure="pm25_vrbanski"),
    "PM25 log scale": make_spec("PM25", transform="log"),
    "PM25 Enota za bolezni only": make_spec("PM25", outcome="arr_disease_mts_all"),
    "PM25 INP only": make_spec("PM25", outcome="arr_inp_mts_all"),
}
sens_models = {k: fit_model(v) for k, v in sens_specs.items()}
sens = pd.DataFrame({k: summarize(m) for k, m in sens_models.items()}).T
sens.to_csv(OUTPUTS / "sensitivity.csv")

q = {}
for h in MODELS:
    ms = {d: sens_models[f"{h} primary"] if d == DF_PER_YEAR else sens_models[f"{h} time {d} df/year"] for d in (4, DF_PER_YEAR, 10)}
    same_days = len({tuple(np.flatnonzero(m["mask"])) for m in ms.values()}) == 1
    phi = ms[10]["res"].scale
    q[h] = {**{f"{d} df/year": qaic(m, phi) for d, m in ms.items()}, "same days": same_days}
print("QAIC by time-spline df (common days and dispersion; lower = better fit; the prespecified 7 df/year is kept):")
display(pd.DataFrame(q).T)

In [ ]:
def forest(ax, labels, est, lo, hi, title):
    y = np.arange(len(labels))[::-1]
    est, lo, hi = (np.asarray(v, dtype=float) for v in (est, lo, hi))
    ax.errorbar(est, y, xerr=[est - lo, hi - est], fmt="o", color=COLORS["blue"], ecolor=COLORS["blue"], ms=4, capsize=0, lw=1.2)
    ax.axvline(1, color="black", lw=0.6)
    ax.set_yticks(y, labels, fontsize=7)
    ax.set_title(title, loc="left")
    ax.set_xlabel("RR (nominal 95% CI)")


sub = lambda h: sens.loc[[k for k in sens.index if k.startswith(h)]]
short = lambda idx: [i.split(" ", 1)[1] for i in idx]
h1, h2, h3, h4 = sub("H1"), sub("H2"), sub("H3"), sub("PM25")
fig, axes = plt.subplots(1, 5, figsize=(19, 6), gridspec_kw={"wspace": 0.95})
forest(axes[0], short(h1.index), h1["RR_heat_lag0"], h1["lo_heat_lag0"], h1["hi_heat_lag0"], "H1 heat, same day\n(99th pct vs median)")
forest(axes[1], short(h1.index), h1["RR_cold_lag0"], h1["lo_cold_lag0"], h1["hi_cold_lag0"], "H1 cold, same day\n(1st pct vs median)")
forest(axes[2], short(h2.index), h2["RR"], h2["lo"], h2["hi"], "H2 influenza, cumulative\n(10th to 90th percentile)")
forest(axes[3], [f"{i}{' (p10→p90)' if inc != 'per 10 µg/m³' else ''}" for i, inc in zip(short(h3.index), h3["increment"])],
       h3["RR"], h3["lo"], h3["hi"], "H3 PM10, cumulative\n(per 10 µg/m³)")
forest(axes[4], [f"{i}{' (p10→p90)' if inc != 'per 10 µg/m³' else ''}" for i, inc in zip(short(h4.index), h4["increment"])],
       h4["RR"], h4["lo"], h4["hi"], "PM2.5 (secondary), cumulative\n(per 10 µg/m³)")
fig.savefig(OUTPUTS / "sensitivity_forest.png", bbox_inches="tight")
plt.show()

## 16. Run record

In [ ]:
def git(*args) -> str:
    try:
        return subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip()
    except Exception:  # noqa: BLE001
        return "unavailable"


PACKAGES = ["pandas", "numpy", "statsmodels", "patsy", "scipy", "matplotlib"]
run_info = {
    "run_at": datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds"),
    "git_commit": git("rev-parse", "HEAD"),
    "git_uncommitted_changes": bool(git("status", "--porcelain")),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "packages": {p: pkg_version(p) for p in PACKAGES},
    "r_version": subprocess.run([shutil.which("Rscript"), "--version"], capture_output=True, text=True).stderr.strip() if shutil.which("Rscript") else None,
    "settings": {"primary": PRIMARY, "secondary": SECONDARY, "defaults": DEFAULTS, "residual_decision": RESID, "alpha": ALPHA,
                 "n_dummy": N_DUMMY, "seed": SEED, "kappa": KAPPA},
    "input_panel_sha256": panel_sha,
    "input_run_02": {"run_at": run_info_02["run_at"], "git_commit": run_info_02["git_commit"]},
    "outputs": {p.name: sha256(p) for p in sorted(OUTPUTS.glob("*.csv"))},
}
(OUTPUTS / "run_info.json").write_text(json.dumps(run_info, indent=2, ensure_ascii=False, default=str))
print(f"Saved results and {(OUTPUTS / 'run_info.json').relative_to(ROOT)}")